# Batch face-only swap (Krea2 model, hair/clothes/background preserved)

Uses the same Krea2 identity-edit model as the head-swap work, but with a **face-only prompt**: only the facial features (bone structure, eyes, nose, mouth, skin) change. Hair, clothing, pose and background are explicitly instructed to stay exactly as they are, and the route is forced to `crop_stitch` -- only a small face crop is ever regenerated and pasted back onto the untouched original, so nothing outside that crop can drift.

Upload 13 body/face pairs and run all of them in one pass.

**3 cells: Setup -> Upload -> Run.**

## 1 · Setup

In [ ]:
from pathlib import Path
import subprocess, os

REPO = Path("/content/headswap_V2")
REPO_URL = "https://github.com/malihashar/headswap_V2.git"
BRANCH = "face-swap-only"

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Runtime -> Change runtime type -> GPU, then Run all."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

from google.colab import drive
drive.mount("/content/drive")

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
subprocess.run(
    ["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True
)

os.chdir(REPO)
# ComfyUI + deps + Krea2 Identity Edit nodes/weights (Drive-cached after first run).
subprocess.run(["bash", "scripts/setup_colab.sh", "--krea2"], check=True, cwd=str(REPO))
print("Setup complete.")


## 2 · Upload 13 pairs

Run this once. It renders **26 upload buttons** -- one Body and one Face per pair. Click each and pick its file; they stay filled in, so you can do them in any order and re-click one to replace it. Nothing is processed here.

When you're done, run cell 3. Pairs with only one side filled in are skipped (so you can test with fewer than 13).

In [ ]:
import ipywidgets as widgets
from IPython.display import display

N_PAIRS = 13

body_uploaders, face_uploaders, rows = [], [], []
for i in range(1, N_PAIRS + 1):
    b = widgets.FileUpload(accept="image/*", multiple=False,
                           description=f"Body {i}")
    f = widgets.FileUpload(accept="image/*", multiple=False,
                           description=f"Face {i}")
    body_uploaders.append(b)
    face_uploaders.append(f)
    rows.append(widgets.HBox(
        [widgets.Label(f"Pair {i:2d}", layout=widgets.Layout(width="70px")), b, f]
    ))

display(widgets.VBox(rows))
print("Fill in the pairs above, then run cell 3.")


## 3 · Run all pairs

Loads the model once, then runs each pair **one at a time**, showing each result as it finishes.

Face-only prompt, `crop_stitch` forced (no full-frame regeneration -- nothing outside the face crop is ever touched), `preserve_expression=True` (keeps the target's own expression), `headwear_prompt_policy` and the hair-replacement clause both off (the base head-swap prompt otherwise explicitly instructs the model to replace hair, which is the opposite of what this needs).

In [ ]:
from pathlib import Path
import sys, os, io, time

REPO = Path("/content/headswap_V2")
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

from headswap.config import load_config
from headswap.pipelines.krea2 import Krea2IdentityEditPipeline
from PIL import Image
from IPython.display import display, Markdown

FACE_SWAP_PROMPT = (
    "Replace only the facial features in the first image with the facial "
    "features from the second image -- bone structure, jawline, eyebrows, "
    "eyes, nose, mouth, and skin -- matching the exact facial identity of "
    "the second person. "
    "Do not change the hair, hairstyle, hairline, hair length, or hair "
    "colour in any way -- keep the first person's own hair exactly as it "
    "already is. "
    "CRITICAL: copy the facial expression from the first image exactly -- "
    "if they are smiling, the result must smile the same way; if they are "
    "not smiling, do not add a smile. Keep mouth shape, smile/no-smile, "
    "eye gaze, and micro-expressions from the first image only -- never "
    "from the second image. "
    "CRITICAL: keep the head facing the exact same direction as the first "
    "image -- do not turn, rotate, or angle the head to the left or right, "
    "and do not tilt it. The eyes must look in the exact same direction as "
    "the first image. Head yaw, pitch, and roll must exactly match the "
    "first image. "
    "Keep the body, clothing, hair, pose, head rotation, camera angle, "
    "lighting, and background from the first image exactly as they are. "
    "Use only the facial identity from the second image -- do not copy "
    "hair, clothing, collar, or shoulders from the second image. If other "
    "people are visible, leave them completely unchanged. Photorealistic, "
    "natural skin texture, sharp details, lighting matched to the first "
    "image."
)

base_cfg = load_config(str(REPO / "configs" / "krea2_identity_edit.yaml"))
cfg = dict(base_cfg)
cfg.update({
    "prompt": FACE_SWAP_PROMPT,
    # The base prompt/reinforcement both say "replace the hair completely" --
    # this route needs the opposite, so both are switched off and the
    # instruction lives entirely in FACE_SWAP_PROMPT above instead.
    "multi_hair_replace_prompt": False,
    # Default headwear-preserve text says "This is a HEAD SWAP, not a face
    # swap: the second person's hair MUST be transferred" -- exactly wrong
    # here. Off entirely; headwear is just part of "everything but the face
    # stays the same," no special instruction needed.
    "headwear_prompt_policy": False,
    "preserve_expression": True,
    # Force crop_stitch unconditionally -- no auto-upgrade to full-frame
    # regeneration for full-body photos, which is what makes clothing/
    # background preservation fragile. crop_stitch only ever touches a
    # small face crop and pastes it back onto the untouched original.
    "enable_body_route": False,
    "enable_lighting_route": False,
    "multi_person_edit_mode": "crop_stitch",
})

CACHE_DIR = Path("/content/drive/MyDrive/headswap_V2/models")
UPLOAD_DIR = Path("/content/face_swap_uploads")
OUT_DIR = Path("/content/face_swap_results")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)


def _bytes_from(uploader):
    """FileUpload.value shape differs between ipywidgets 7 and 8."""
    val = uploader.value
    if not val:
        return None
    if isinstance(val, (tuple, list)):          # ipywidgets 8
        return bytes(val[0]["content"])
    name = next(iter(val))                       # ipywidgets 7
    return bytes(val[name]["content"])


pairs, half_filled = [], []
for i, (bu, fu) in enumerate(zip(body_uploaders, face_uploaders), start=1):
    bdata, fdata = _bytes_from(bu), _bytes_from(fu)
    if bdata is None and fdata is None:
        continue
    if bdata is None or fdata is None:
        half_filled.append(i)
        continue
    bp, fp = UPLOAD_DIR / f"body_{i:02d}.png", UPLOAD_DIR / f"face_{i:02d}.png"
    Image.open(io.BytesIO(bdata)).convert("RGB").save(bp)
    Image.open(io.BytesIO(fdata)).convert("RGB").save(fp)
    pairs.append((i, bp, fp))

if half_filled:
    print(f"Skipping pairs with only one side uploaded: {half_filled}")
if not pairs:
    raise SystemExit("No complete pairs -- fill in the buttons in cell 2 first.")

print(f"{len(pairs)} complete pair(s). Loading model once, then running "
      "them one at a time.\n")

pipe = Krea2IdentityEditPipeline(cfg=cfg, cache_dir=CACHE_DIR)

t0 = time.perf_counter()
for n, (idx, body_path, face_path) in enumerate(pairs, start=1):
    body = Image.open(body_path).convert("RGB")
    face = Image.open(face_path).convert("RGB")
    pair_out = OUT_DIR / f"pair_{idx:02d}"
    t = time.perf_counter()
    res = pipe.run(body, face, out_dir=pair_out)
    display(Markdown(
        f"### Pair {idx} &nbsp;·&nbsp; {time.perf_counter() - t:.0f}s "
        f"&nbsp;·&nbsp; [{n}/{len(pairs)}]"
    ))
    display(res.image)

print(f"\nAll {len(pairs)} pair(s) done in {time.perf_counter() - t0:.0f}s total.")
print(f"Saved under {OUT_DIR}")
